In [ ]:
import pandas as pd
import numpy as np
import ast

IN_PATH = "Claude_multi_step.csv"
OUT_PATH = "Claude_multi_step_extracted.csv"

PRED_COL = "predicted"
GT_COL   = "ground_truth"

KEYS = ["TOTALDEMAND", "RRP"]


def safe_to_dict(x):
    """Convert a cell to dict safely (handles dict, stringified dict, NaN)."""
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, dict) else {}
        except Exception:
            return {}
    return {}


def extract_series(d, key):
    """
    Extract all numeric first elements from dict[key] where dict[key] can be:
      - [value, ts]
      - [[value, ts]]
      - [[v1, ts], [v2, ts], ...]
    Returns: list[float] (possibly empty)
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # Case A: [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 1 and not isinstance(v[0], (list, tuple)):
        # v[0] is the numeric value
        try:
            return [float(v[0])]
        except Exception:
            return []

    # Case B/C: [[value, ts], [value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                try:
                    out.append(float(item[0]))
                except Exception:
                    # skip non-numeric
                    pass
        return out

    return []


def extract_timestamp_series(d, key):
    """
    Optional: extract all timestamps (2nd element) if present.
    Returns: list[str]
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 2 and not isinstance(v[0], (list, tuple)):
        return [str(v[1])]

    # [[value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append(str(item[1]))
        return out

    return []


# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(IN_PATH)

# Ensure dicts
df[PRED_COL] = df[PRED_COL].apply(safe_to_dict)
df[GT_COL]   = df[GT_COL].apply(safe_to_dict)

# ----------------------------
# Extract series (values)
# ----------------------------
for k in KEYS:
    df[f"pred_{k}_series"] = df[PRED_COL].apply(lambda d: extract_series(d, k))
    df[f"gt_{k}_series"]   = df[GT_COL].apply(lambda d: extract_series(d, k))

# Optional: extract timestamp series too
for k in KEYS:
    df[f"pred_{k}_ts_series"] = df[PRED_COL].apply(lambda d: extract_timestamp_series(d, k))
    df[f"gt_{k}_ts_series"]   = df[GT_COL].apply(lambda d: extract_timestamp_series(d, k))

# ----------------------------
# (Optional) Convenience: scalar value if exactly one step
# ----------------------------
def single_or_nan(xs):
    return xs[0] if isinstance(xs, list) and len(xs) == 1 else np.nan

for k in KEYS:
    df[f"pred_{k}_single"] = df[f"pred_{k}_series"].apply(single_or_nan)
    df[f"gt_{k}_single"]   = df[f"gt_{k}_series"].apply(single_or_nan)

# ----------------------------
# Diagnostics
# ----------------------------
print("Non-empty series counts:")
for k in KEYS:
    n_pred = (df[f"pred_{k}_series"].apply(len) > 0).sum()
    n_gt   = (df[f"gt_{k}_series"].apply(len) > 0).sum()
    print(f"  {k}: pred={n_pred} rows, gt={n_gt} rows")

print("\nPreview of extracted columns:")
cols_preview = []
for k in KEYS:
    cols_preview += [f"pred_{k}_series", f"gt_{k}_series"]
print(df[cols_preview].head(5))

# ----------------------------
# Save
# ----------------------------
df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


In [ ]:
import pandas as pd
import numpy as np
import ast

IN_PATH = "Gemini_multi_step.csv"
OUT_PATH = "Gemini_multi_step_extracted.csv"

PRED_COL = "predicted"
GT_COL   = "ground_truth"

KEYS = ["TOTALDEMAND", "RRP"]


def safe_to_dict(x):
    """Convert a cell to dict safely (handles dict, stringified dict, NaN)."""
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, dict) else {}
        except Exception:
            return {}
    return {}


def extract_series(d, key):
    """
    Extract all numeric first elements from dict[key] where dict[key] can be:
      - [value, ts]
      - [[value, ts]]
      - [[v1, ts], [v2, ts], ...]
    Returns: list[float] (possibly empty)
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # Case A: [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 1 and not isinstance(v[0], (list, tuple)):
        # v[0] is the numeric value
        try:
            return [float(v[0])]
        except Exception:
            return []

    # Case B/C: [[value, ts], [value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                try:
                    out.append(float(item[0]))
                except Exception:
                    # skip non-numeric
                    pass
        return out

    return []


def extract_timestamp_series(d, key):
    """
    Optional: extract all timestamps (2nd element) if present.
    Returns: list[str]
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 2 and not isinstance(v[0], (list, tuple)):
        return [str(v[1])]

    # [[value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append(str(item[1]))
        return out

    return []


# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(IN_PATH)

# Ensure dicts
df[PRED_COL] = df[PRED_COL].apply(safe_to_dict)
df[GT_COL]   = df[GT_COL].apply(safe_to_dict)

# ----------------------------
# Extract series (values)
# ----------------------------
for k in KEYS:
    df[f"pred_{k}_series"] = df[PRED_COL].apply(lambda d: extract_series(d, k))
    df[f"gt_{k}_series"]   = df[GT_COL].apply(lambda d: extract_series(d, k))

# Optional: extract timestamp series too
for k in KEYS:
    df[f"pred_{k}_ts_series"] = df[PRED_COL].apply(lambda d: extract_timestamp_series(d, k))
    df[f"gt_{k}_ts_series"]   = df[GT_COL].apply(lambda d: extract_timestamp_series(d, k))

# ----------------------------
# (Optional) Convenience: scalar value if exactly one step
# ----------------------------
def single_or_nan(xs):
    return xs[0] if isinstance(xs, list) and len(xs) == 1 else np.nan

for k in KEYS:
    df[f"pred_{k}_single"] = df[f"pred_{k}_series"].apply(single_or_nan)
    df[f"gt_{k}_single"]   = df[f"gt_{k}_series"].apply(single_or_nan)

# ----------------------------
# Diagnostics
# ----------------------------
print("Non-empty series counts:")
for k in KEYS:
    n_pred = (df[f"pred_{k}_series"].apply(len) > 0).sum()
    n_gt   = (df[f"gt_{k}_series"].apply(len) > 0).sum()
    print(f"  {k}: pred={n_pred} rows, gt={n_gt} rows")

print("\nPreview of extracted columns:")
cols_preview = []
for k in KEYS:
    cols_preview += [f"pred_{k}_series", f"gt_{k}_series"]
print(df[cols_preview].head(5))

# ----------------------------
# Save
# ----------------------------
df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


In [ ]:
import pandas as pd
import numpy as np
import ast

IN_PATH = "OpenAI_multi_step.csv"
OUT_PATH = "OpenAI_multi_step_extracted.csv"

PRED_COL = "predicted"
GT_COL   = "ground_truth"

KEYS = ["TOTALDEMAND", "RRP"]


def safe_to_dict(x):
    """Convert a cell to dict safely (handles dict, stringified dict, NaN)."""
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, dict) else {}
        except Exception:
            return {}
    return {}


def extract_series(d, key):
    """
    Extract all numeric first elements from dict[key] where dict[key] can be:
      - [value, ts]
      - [[value, ts]]
      - [[v1, ts], [v2, ts], ...]
    Returns: list[float] (possibly empty)
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # Case A: [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 1 and not isinstance(v[0], (list, tuple)):
        # v[0] is the numeric value
        try:
            return [float(v[0])]
        except Exception:
            return []

    # Case B/C: [[value, ts], [value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                try:
                    out.append(float(item[0]))
                except Exception:
                    # skip non-numeric
                    pass
        return out

    return []


def extract_timestamp_series(d, key):
    """
    Optional: extract all timestamps (2nd element) if present.
    Returns: list[str]
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 2 and not isinstance(v[0], (list, tuple)):
        return [str(v[1])]

    # [[value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append(str(item[1]))
        return out

    return []


# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(IN_PATH)

# Ensure dicts
df[PRED_COL] = df[PRED_COL].apply(safe_to_dict)
df[GT_COL]   = df[GT_COL].apply(safe_to_dict)

# ----------------------------
# Extract series (values)
# ----------------------------
for k in KEYS:
    df[f"pred_{k}_series"] = df[PRED_COL].apply(lambda d: extract_series(d, k))
    df[f"gt_{k}_series"]   = df[GT_COL].apply(lambda d: extract_series(d, k))

# Optional: extract timestamp series too
for k in KEYS:
    df[f"pred_{k}_ts_series"] = df[PRED_COL].apply(lambda d: extract_timestamp_series(d, k))
    df[f"gt_{k}_ts_series"]   = df[GT_COL].apply(lambda d: extract_timestamp_series(d, k))

# ----------------------------
# (Optional) Convenience: scalar value if exactly one step
# ----------------------------
def single_or_nan(xs):
    return xs[0] if isinstance(xs, list) and len(xs) == 1 else np.nan

for k in KEYS:
    df[f"pred_{k}_single"] = df[f"pred_{k}_series"].apply(single_or_nan)
    df[f"gt_{k}_single"]   = df[f"gt_{k}_series"].apply(single_or_nan)

# ----------------------------
# Diagnostics
# ----------------------------
print("Non-empty series counts:")
for k in KEYS:
    n_pred = (df[f"pred_{k}_series"].apply(len) > 0).sum()
    n_gt   = (df[f"gt_{k}_series"].apply(len) > 0).sum()
    print(f"  {k}: pred={n_pred} rows, gt={n_gt} rows")

print("\nPreview of extracted columns:")
cols_preview = []
for k in KEYS:
    cols_preview += [f"pred_{k}_series", f"gt_{k}_series"]
print(df[cols_preview])

# ----------------------------
# Save
# ----------------------------
df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


In [ ]:
import pandas as pd
import numpy as np
import ast

IN_PATH = "DeepSeek_multi_step.csv"
OUT_PATH = "DeepSeek_multi_step_extracted.csv"

PRED_COL = "predicted"
GT_COL   = "ground_truth"

KEYS = ["TOTALDEMAND", "RRP"]


def safe_to_dict(x):
    """Convert a cell to dict safely (handles dict, stringified dict, NaN)."""
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            val = ast.literal_eval(x)
            return val if isinstance(val, dict) else {}
        except Exception:
            return {}
    return {}


def extract_series(d, key):
    """
    Extract all numeric first elements from dict[key] where dict[key] can be:
      - [value, ts]
      - [[value, ts]]
      - [[v1, ts], [v2, ts], ...]
    Returns: list[float] (possibly empty)
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # Case A: [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 1 and not isinstance(v[0], (list, tuple)):
        # v[0] is the numeric value
        try:
            return [float(v[0])]
        except Exception:
            return []

    # Case B/C: [[value, ts], [value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 1:
                try:
                    out.append(float(item[0]))
                except Exception:
                    # skip non-numeric
                    pass
        return out

    return []


def extract_timestamp_series(d, key):
    """
    Optional: extract all timestamps (2nd element) if present.
    Returns: list[str]
    """
    if not isinstance(d, dict) or key not in d:
        return []

    v = d[key]

    # [value, ts]
    if isinstance(v, (list, tuple)) and len(v) >= 2 and not isinstance(v[0], (list, tuple)):
        return [str(v[1])]

    # [[value, ts], ...]
    if isinstance(v, (list, tuple)):
        out = []
        for item in v:
            if isinstance(item, (list, tuple)) and len(item) >= 2:
                out.append(str(item[1]))
        return out

    return []


# ----------------------------
# Load
# ----------------------------
df = pd.read_csv(IN_PATH)

# Ensure dicts
df[PRED_COL] = df[PRED_COL].apply(safe_to_dict)
df[GT_COL]   = df[GT_COL].apply(safe_to_dict)

# ----------------------------
# Extract series (values)
# ----------------------------
for k in KEYS:
    df[f"pred_{k}_series"] = df[PRED_COL].apply(lambda d: extract_series(d, k))
    df[f"gt_{k}_series"]   = df[GT_COL].apply(lambda d: extract_series(d, k))

# Optional: extract timestamp series too
for k in KEYS:
    df[f"pred_{k}_ts_series"] = df[PRED_COL].apply(lambda d: extract_timestamp_series(d, k))
    df[f"gt_{k}_ts_series"]   = df[GT_COL].apply(lambda d: extract_timestamp_series(d, k))

# ----------------------------
# (Optional) Convenience: scalar value if exactly one step
# ----------------------------
def single_or_nan(xs):
    return xs[0] if isinstance(xs, list) and len(xs) == 1 else np.nan

for k in KEYS:
    df[f"pred_{k}_single"] = df[f"pred_{k}_series"].apply(single_or_nan)
    df[f"gt_{k}_single"]   = df[f"gt_{k}_series"].apply(single_or_nan)

# ----------------------------
# Diagnostics
# ----------------------------
print("Non-empty series counts:")
for k in KEYS:
    n_pred = (df[f"pred_{k}_series"].apply(len) > 0).sum()
    n_gt   = (df[f"gt_{k}_series"].apply(len) > 0).sum()
    print(f"  {k}: pred={n_pred} rows, gt={n_gt} rows")

print("\nPreview of extracted columns:")
cols_preview = []
for k in KEYS:
    cols_preview += [f"pred_{k}_series", f"gt_{k}_series"]
print(df[cols_preview].head(5))

# ----------------------------
# Save
# ----------------------------
df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)
